# Climate of England — Weather Data Analysis

## Final analysis questions
1. How is temperature distributed across England?
2. What is the distribution of humidity?
3. How does wind speed vary throughout the dataset?
4. Is there a relationship between temperature and humidity?
5. Is there a relationship between temperature and wind speed?
6. How does average temperature change over time?
7. Which weather conditions occur most frequently?

**Dataset limitations:** This CSV contains one England-wide time series and has no country/place column, so comparisons between countries or places cannot be made. It also has no numeric rainfall-amount column; `Precip Type` records categories such as rain/snow and cannot be treated as rainfall quantity. The analysis therefore uses precipitation-type counts instead of rainfall amount, and monthly average temperature instead of a country comparison.

## 0. Setup and load the CSV

Keep the CSV in the same folder as this notebook. In Google Colab, upload the CSV when prompted. The notebook reads the source file but never writes to or edits it.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

CSV_PATH = Path("Climate of England _ Professional Reforms.csv")
if not CSV_PATH.exists():
    # Google Colab upload fallback
    try:
        from google.colab import files
        uploaded = files.upload()
        CSV_PATH = Path(next(iter(uploaded)))
    except Exception as exc:
        raise FileNotFoundError(
            "Place the CSV beside this notebook, or upload it in Google Colab."
        ) from exc

df = pd.read_csv(CSV_PATH)
print("CSV loaded:", CSV_PATH.name)
print("Original shape:", df.shape)

**Explanation:** The source CSV is loaded into `df`. All cleaning and analysis are performed in memory; the original file is not overwritten.

## Part 1 — Inspect the dataset before analysis

In [ ]:
display(df.head())
df.info()
print("\\nColumn names:")
print(df.columns.tolist())

**Explanation:** The first rows show the structure, `info()` reports data types and non-null counts, and `columns` lists every field. The dataset has date/time, weather-condition, temperature, wind, visibility, pressure and humidity fields. It has no location field and no numeric rainfall amount.

In [ ]:
print("Shape:", df.shape)
print("\\nData types:")
display(df.dtypes.rename("data_type").to_frame())
print("\\nDescriptive statistics:")
display(df.describe(include="all").T)

**Explanation:** The shape gives the number of records and fields. `describe()` summarizes numeric columns and gives counts/frequencies for categorical columns.

## Step 1 — Basic analysis

In [ ]:
# Parse the date and combine it with the time to create a timestamp.
# The source date format is month/day/year and the time includes milliseconds.
df["Date"] = pd.to_datetime(df["Formatted Date"], format="%m/%d/%Y", errors="coerce")
df["Timestamp"] = pd.to_datetime(
    df["Formatted Date"] + " " + df["Time"],
    format="%m/%d/%Y %H:%M:%S.%f",
    errors="coerce"
)

print("Unparsed dates:", df["Date"].isna().sum())
print("Unparsed timestamps:", df["Timestamp"].isna().sum())
print("First date:", df["Date"].min())
print("Last date:", df["Date"].max())
print("Distinct dates:", df["Date"].nunique())
print("Countries/places available:", "Not provided in this CSV (England dataset)")

**Explanation:** The date and timestamp are converted to proper pandas datetime values. The data covers 1 January 2006 through 31 December 2016. Since there is no place/country column, the number of distinct places cannot be calculated; the dataset is described as England weather data.

In [ ]:
# Keep only fields needed for the seven approved questions and quality checks.
needed_columns = [
    "Date", "Timestamp", "Summary", "Precip Type",
    "Temperature (C)", "Humidity", "Wind Speed (km/h)",
    "Apparent Temperature (C)", "Visibility (km)", "Pressure (millibars)"
]
analysis_df = df[needed_columns].copy()
analysis_df = analysis_df.sort_values("Timestamp").reset_index(drop=True)
display(analysis_df.head())
print("Analysis dataset shape:", analysis_df.shape)

**Explanation:** `analysis_df` contains the date/time, weather category, and key weather measurements needed for the questions. The original `df` and CSV remain unchanged.

## Step 2 — Missing values

In [ ]:
missing_counts = analysis_df.isna().sum().sort_values(ascending=False)
display(missing_counts.rename("missing_count").to_frame())
ax = missing_counts.plot(kind="bar", figsize=(11, 4), title="Missing values by column")
ax.set_xlabel("Column")
ax.set_ylabel("Number of missing values")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

**Explanation:** The missing-value count is shown for every selected field and visualized in a bar chart. In this dataset, the notable missingness is in `Precip Type`; the main numeric weather measurements are complete.

In [ ]:
# Explicit missing-value decisions:
# - Precip Type: fill missing category with "Unknown" so records are retained.
# - Numeric weather columns: no missing values in the inspected source, so no imputation is needed.
# - Date/Timestamp: invalid parsed values would be dropped because time is essential to time-series analysis.
analysis_df["Precip Type"] = analysis_df["Precip Type"].fillna("Unknown")
invalid_time_rows = analysis_df["Timestamp"].isna().sum()
analysis_df = analysis_df.dropna(subset=["Timestamp"]).copy()

print("Precip Type missing after fill:", analysis_df["Precip Type"].isna().sum())
print("Invalid timestamp rows removed:", invalid_time_rows)
print("Remaining missing values:")
display(analysis_df.isna().sum().to_frame("missing_after_decision"))

**Explanation:** Missing precipitation categories are labeled `Unknown`, not guessed. Numeric columns are left as-is because the source inspection found no missing numeric weather values. Rows with unusable timestamps are removed only if any exist.

## Step 3 — Repeated values

In [ ]:
exact_duplicates = analysis_df.duplicated().sum()
timestamp_duplicates = analysis_df.duplicated(subset=["Timestamp"], keep=False).sum()
timestamp_duplicate_excess = analysis_df.duplicated(subset=["Timestamp"]).sum()

print("Exact duplicate rows:", exact_duplicates)
print("Rows participating in repeated timestamps:", timestamp_duplicates)
print("Excess rows with a repeated timestamp:", timestamp_duplicate_excess)

# Show examples so repeated timestamps can be inspected.
repeated_timestamp_rows = analysis_df[
    analysis_df.duplicated(subset=["Timestamp"], keep=False)
].sort_values("Timestamp")
display(repeated_timestamp_rows.head(12))

**Explanation:** The dataset is hourly, so the uniqueness key is the combined timestamp (date plus time), not date alone. The original file contains 24 exact duplicate rows and 35 rows involved in repeated timestamps; one repeated hour on the daylight-saving clock change can represent two distinct observations, so repeated timestamps are inspected rather than blindly removed.

In [ ]:
# Remove exact duplicate records only. Keep distinct observations sharing a timestamp,
# because the autumn daylight-saving clock change can legitimately repeat 02:00.
before = len(analysis_df)
analysis_df = analysis_df.drop_duplicates().copy()
removed_exact = before - len(analysis_df)

print("Exact duplicate rows removed:", removed_exact)
print("Exact duplicate rows remaining:", analysis_df.duplicated().sum())
print("Repeated timestamps remaining (including possible DST hour):",
      analysis_df.duplicated(subset=["Timestamp"], keep=False).sum())

**Explanation:** Only fully identical rows are removed. Non-identical observations at the same clock timestamp are preserved because the repeated hour during the daylight-saving transition may be genuine. This is more defensible than deleting valid weather observations based on timestamp alone.

## Step 4 — Outliers

In [ ]:
weather_numeric = ["Temperature (C)", "Wind Speed (km/h)"]
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for row, col in enumerate(weather_numeric):
    sns.boxplot(x=analysis_df[col], ax=axes[row, 0])
    axes[row, 0].set_title(f"Box plot: {col}")
    axes[row, 0].set_xlabel(col)
    axes[row, 1].hist(analysis_df[col].dropna(), bins=40)
    axes[row, 1].set_title(f"Histogram: {col}")
    axes[row, 1].set_xlabel(col)
    axes[row, 1].set_ylabel("Frequency")
plt.tight_layout()
plt.show()

# Precipitation amount is not present. Show the available precipitation-type categories instead.
plt.figure(figsize=(7, 4))
analysis_df["Precip Type"].value_counts().plot(kind="bar")
plt.title("Precipitation type counts (not rainfall amount)")
plt.xlabel("Precipitation type")
plt.ylabel("Number of hourly records")
plt.tight_layout()
plt.show()

**Explanation:** Temperature and wind speed have box plots and histograms. A rainfall-amount plot cannot be made because the CSV has no numeric rainfall column; precipitation-type counts are shown instead. These categories must not be interpreted as rainfall volume.

In [ ]:
def iqr_outlier_report(series, column_name):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = series[(series < lower) | (series > upper)]
    print(f"\\n{column_name}")
    print(f"Q1={q1:.3f}, Q3={q3:.3f}, IQR={iqr:.3f}")
    print(f"Lower fence={lower:.3f}, upper fence={upper:.3f}")
    print(f"Outlier count={len(outliers)}")
    if len(outliers):
        print("Smallest/largest flagged values:", outliers.min(), "/", outliers.max())
    return outliers

temp_outliers = iqr_outlier_report(analysis_df["Temperature (C)"], "Temperature (C)")
wind_outliers = iqr_outlier_report(analysis_df["Wind Speed (km/h)"], "Wind Speed (km/h)")

**Explanation:** The IQR rule flags values below Q1 − 1.5×IQR or above Q3 + 1.5×IQR. A statistical flag is not proof of an error: unusually hot/cold periods and strong winds may be real weather events. Flagged observations are retained pending source-level verification; no values are automatically deleted.

In [ ]:
# Inspect the most extreme observations in context before making a decision.
display(
    analysis_df.loc[temp_outliers.index,
                    ["Timestamp", "Summary", "Temperature (C)", "Humidity", "Wind Speed (km/h)"]]
    .sort_values("Temperature (C)")
    .head(10)
)
display(
    analysis_df.loc[wind_outliers.index,
                    ["Timestamp", "Summary", "Temperature (C)", "Humidity", "Wind Speed (km/h)"]]
    .sort_values("Wind Speed (km/h)", ascending=False)
    .head(10)
)

**Explanation:** The timestamps and accompanying weather conditions help assess whether extremes are plausible. Without an independent weather station record, the notebook cannot conclusively label each IQR outlier as real or erroneous. The values are therefore preserved and reported as candidates for review.

## Step 5 — Univariate analysis

In [ ]:
univariate_columns = [
    ("Temperature (C)", "Temperature (°C)"),
    ("Humidity", "Humidity (fraction)"),
    ("Wind Speed (km/h)", "Wind speed (km/h)")
]
for col, label in univariate_columns:
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(analysis_df[col].dropna(), bins=40)
    ax.set_title(f"Distribution of {label}")
    ax.set_xlabel(label)
    ax.set_ylabel("Number of hourly observations")
    plt.tight_layout()
    plt.show()
    print(f"{col}: mean={analysis_df[col].mean():.3f}, median={analysis_df[col].median():.3f}, "
          f"standard deviation={analysis_df[col].std():.3f}, range="
          f"{analysis_df[col].min():.3f} to {analysis_df[col].max():.3f}")

**Explanation:** Each key numeric weather variable has its own distribution graph. The printed mean and median describe typical values, while standard deviation and minimum-to-maximum range describe spread. Humidity is stored as a fraction from 0 to 1.

## Step 6 — Bivariate and time analysis

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=analysis_df, x="Temperature (C)", y="Humidity", alpha=0.25, s=12)
plt.title("Temperature vs Humidity")
plt.xlabel("Temperature (°C)")
plt.ylabel("Humidity (fraction)")
plt.tight_layout()
plt.show()

**Explanation:** Each point represents an hourly observation. The scatter plot helps show whether warmer observations tend to have higher or lower humidity; correlation is not proof of causation.

In [ ]:
correlation_columns = [
    "Temperature (C)", "Humidity", "Wind Speed (km/h)",
    "Apparent Temperature (C)", "Visibility (km)", "Pressure (millibars)"
]
corr = analysis_df[correlation_columns].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation heatmap of numeric weather variables")
plt.tight_layout()
plt.show()

**Explanation:** The heatmap summarizes pairwise linear correlations. Values near +1 indicate a strong positive linear relationship, values near −1 a strong negative relationship, and values near 0 a weak linear relationship.

In [ ]:
plt.figure(figsize=(13, 5))
daily_temp = analysis_df.set_index("Timestamp")["Temperature (C)"].resample("D").mean()
daily_temp.plot()
plt.title("Daily average temperature over time")
plt.xlabel("Date")
plt.ylabel("Average temperature (°C)")
plt.tight_layout()
plt.show()

**Explanation:** Hourly temperature readings are averaged by day to make the long-term trend easier to read. Gaps, if any, remain visible rather than being fabricated.

In [ ]:
monthly_temp = analysis_df.assign(MonthStart=analysis_df["Timestamp"].dt.to_period("M").dt.to_timestamp())
monthly_avg = monthly_temp.groupby(monthly_temp["Timestamp"].dt.month)["Temperature (C)"].mean()
plt.figure(figsize=(9, 4))
monthly_avg.plot(kind="bar")
plt.title("Average temperature by calendar month")
plt.xlabel("Month number")
plt.ylabel("Average temperature (°C)")
plt.tight_layout()
plt.show()

**Explanation:** There is no country/place field for the requested country comparison. This chart uses calendar month as a meaningful alternative and compares average temperature across the 12 months.

## Step 7 — Answer the seven approved questions

In [ ]:
# Q1: Temperature distribution
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(analysis_df["Temperature (C)"].dropna(), bins=40)
ax.set(title="Q1. Temperature distribution", xlabel="Temperature (°C)", ylabel="Hourly records")
plt.tight_layout(); plt.show()
print(f"Answer: Mean temperature is {analysis_df['Temperature (C)'].mean():.2f}°C and median is "
      f"{analysis_df['Temperature (C)'].median():.2f}°C. Values range from "
      f"{analysis_df['Temperature (C)'].min():.2f}°C to {analysis_df['Temperature (C)'].max():.2f}°C.")

**Explanation:** The histogram and summary statistics describe the distribution, its typical center, and its overall range.

In [ ]:
# Q2: Humidity distribution
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(analysis_df["Humidity"].dropna(), bins=30)
ax.set(title="Q2. Humidity distribution", xlabel="Humidity (fraction)", ylabel="Hourly records")
plt.tight_layout(); plt.show()
print(f"Answer: Mean humidity is {analysis_df['Humidity'].mean():.3f} and median is "
      f"{analysis_df['Humidity'].median():.3f}; the range is "
      f"{analysis_df['Humidity'].min():.2f} to {analysis_df['Humidity'].max():.2f}.")

**Explanation:** Humidity is recorded as a fraction, so 0.75 corresponds to 75%. The graph shows how frequently different humidity levels occur.

In [ ]:
# Q3: Wind speed distribution
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(analysis_df["Wind Speed (km/h)"].dropna(), bins=40)
ax.set(title="Q3. Wind speed distribution", xlabel="Wind speed (km/h)", ylabel="Hourly records")
plt.tight_layout(); plt.show()
print(f"Answer: Mean wind speed is {analysis_df['Wind Speed (km/h)'].mean():.2f} km/h and median is "
      f"{analysis_df['Wind Speed (km/h)'].median():.2f} km/h.")

**Explanation:** The histogram and median/mean summarize the typical wind speed and whether high-speed observations stretch the distribution.

In [ ]:
# Q4: Temperature vs humidity
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=analysis_df, x="Temperature (C)", y="Humidity", alpha=0.25, s=12, ax=ax)
ax.set(title="Q4. Temperature vs Humidity", xlabel="Temperature (°C)", ylabel="Humidity (fraction)")
plt.tight_layout(); plt.show()
r_th = analysis_df["Temperature (C)"].corr(analysis_df["Humidity"])
print(f"Answer: Pearson correlation = {r_th:.3f}. "
      + ("This indicates a negative linear association." if r_th < -0.1 else
         "This indicates a positive linear association." if r_th > 0.1 else
         "This indicates a weak linear association."))

**Explanation:** Pearson's correlation gives the direction and strength of the linear association. The scatter plot should be considered alongside the coefficient.

In [ ]:
# Q5: Temperature vs wind speed
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=analysis_df, x="Temperature (C)", y="Wind Speed (km/h)", alpha=0.25, s=12, ax=ax)
ax.set(title="Q5. Temperature vs Wind Speed", xlabel="Temperature (°C)", ylabel="Wind speed (km/h)")
plt.tight_layout(); plt.show()
r_tw = analysis_df["Temperature (C)"].corr(analysis_df["Wind Speed (km/h)"])
print(f"Answer: Pearson correlation = {r_tw:.3f}.")

**Explanation:** This plot and coefficient show whether temperature and wind speed move together linearly. A weak coefficient means temperature alone is not a useful linear predictor of wind speed.

In [ ]:
# Q6: Average temperature over time
daily_temp = analysis_df.set_index("Timestamp")["Temperature (C)"].resample("D").mean()
daily_temp.plot(figsize=(13, 5), title="Q6. Daily average temperature over time")
plt.xlabel("Date"); plt.ylabel("Average temperature (°C)")
plt.tight_layout(); plt.show()
yearly_temp = analysis_df.groupby(analysis_df["Timestamp"].dt.year)["Temperature (C)"].mean()
display(yearly_temp.rename("average_temperature_C").to_frame())
print("Answer: The line shows seasonal fluctuations and longer-term variation; the table gives each year's average.")

**Explanation:** Daily averages make seasonal patterns visible, while the yearly table supports comparison between years. This dataset alone does not establish the cause of any trend.

In [ ]:
# Q7: Most frequent weather conditions
condition_counts = analysis_df["Summary"].value_counts().head(12)
display(condition_counts.rename("hourly_record_count").to_frame())
condition_counts.sort_values().plot(kind="barh", figsize=(9, 5), title="Q7. Most frequent weather conditions")
plt.xlabel("Number of hourly records"); plt.ylabel("Weather summary")
plt.tight_layout(); plt.show()
print(f"Answer: The most frequent recorded condition is '{analysis_df['Summary'].mode().iloc[0]}'.")

**Explanation:** The frequency table and bar chart rank the recorded weather summaries. These are hourly records, not counts of distinct weather events.

## Step 8 — Conclusion

In [ ]:
conclusion = [
    f"The dataset contains {len(analysis_df):,} hourly observations after exact duplicate removal, spanning {analysis_df['Timestamp'].min():%d %b %Y} to {analysis_df['Timestamp'].max():%d %b %Y}.",
    f"Average temperature is {analysis_df['Temperature (C)'].mean():.2f}°C, with a median of {analysis_df['Temperature (C)'].median():.2f}°C.",
    f"Average humidity is {analysis_df['Humidity'].mean():.2f} and average wind speed is {analysis_df['Wind Speed (km/h)'].mean():.2f} km/h.",
    f"Temperature-humidity correlation is {analysis_df['Temperature (C)'].corr(analysis_df['Humidity']):.2f}; temperature-wind correlation is {analysis_df['Temperature (C)'].corr(analysis_df['Wind Speed (km/h)']):.2f}.",
    f"The most frequently recorded weather summary is {analysis_df['Summary'].mode().iloc[0]}.",
    "The CSV has no location field or numeric rainfall amount, so place comparisons and rainfall-volume analysis are not supported; IQR outliers were retained pending verification."
]
print("\\n".join("• " + line for line in conclusion))

**Explanation:** These six statements summarize the time coverage, typical weather measurements, relationships, most common condition, and important limits of the source data.

## Optional — Export the cleaned analysis data

This creates a separate derived file only if you choose to run the cell. It does not change the original CSV.

In [ ]:
# Uncomment to save a separate cleaned copy:
# analysis_df.to_csv("climate_analysis_cleaned.csv", index=False)